# 01 · Exploratory analysis

SG2 ranked solo/duo, Emerald+, last ~3 patches. Draft rows come from match details; game-state rows are the subset with a 15-minute timeline.

In [ ]:
import json, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
import sys; sys.path.insert(0, str(ROOT))
DATA = Path(os.environ.get("LOLWP_DATA_DIR", ROOT / "data"))
ART = Path(os.environ.get("LOLWP_ARTIFACTS_DIR", ROOT / "artifacts"))
sns.set_theme(style="whitegrid")
from lolwp.config import STATE_FEATURES
draft = pd.read_parquet(DATA / "draft.parquet")
state = pd.read_parquet(DATA / "state.parquet")
insights = json.load(open(ART / "insights.json"))
len(draft), len(state)

## Rows per patch

In [ ]:
pd.crosstab(draft.patch, draft.split).join(state.groupby("patch").size().rename("state_rows"))

**Interpretation (write 3–5 sentences after running on the real data):** _How many matches per patch? Is the test patch big enough (≥1,500 draft rows) to trust its metrics?_

## Blue-side win rate

In [ ]:
insights["blue_side"]

**Interpretation (write 3–5 sentences after running on the real data):** _Is Blue side's edge real (p-value) and how large is it in percentage points?_

## Win rate vs gold lead at 15:00

In [ ]:
curve = pd.DataFrame(insights["gold_curve"])
ax = curve.plot(x="gold_bin_center", y="win_rate", marker="o", legend=False, figsize=(8, 4))
ax.axhline(0.5, color="grey", ls="--"); ax.set(xlabel="Blue gold lead at 15:00", ylabel="Blue win rate")
plt.show()

**Interpretation (write 3–5 sentences after running on the real data):** _How steep is the curve? What gold lead makes a game roughly 70/30?_

## Win rate by first objectives

In [ ]:
firsts = ["first_blood_blue", "first_dragon_blue", "first_tower_blue"]
pd.DataFrame({f: state.groupby(f).blue_win.mean() for f in firsts}).rename(index={-1: "Red", 0: "None", 1: "Blue"})

**Interpretation (write 3–5 sentences after running on the real data):** _Which first objective is most associated with winning? Remember these overlap with gold lead._

## Correlation between state features

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(state[STATE_FEATURES].corr(), cmap="RdBu_r", center=0)
plt.show()

**Interpretation (write 3–5 sentences after running on the real data):** _Which features are near-duplicates (gold, XP, CS)? What does that mean for reading single coefficients?_

## Champion win rates with 95% intervals

In [ ]:
champs = pd.DataFrame(insights["champions"])
show = pd.concat([champs.head(10), champs.tail(10)])
fig, ax = plt.subplots(figsize=(8, 7))
ax.errorbar(show.win_rate, show.name, xerr=[show.win_rate - show.ci_low, show.ci_high - show.win_rate], fmt="o")
ax.axvline(0.5, color="grey", ls="--"); ax.set_xlabel("Win rate (train split)")
plt.show()

**Interpretation (write 3–5 sentences after running on the real data):** _How many champions' intervals exclude 50%? Why does smoothing toward 50% matter for low-game champions?_